# 6. Hyperparameter Tuning — YOLO Genetic Search

Runs YOLO's built-in evolutionary hyperparameter optimizer (`model.tune()`).
It searches over ~25 parameters (learning rates, augmentation strengths, anchor ratios,
momentum, etc.) using a genetic algorithm and saves the best configuration to `args.yaml`.

Use this notebook **after** a baseline training run to find better hyperparameters
for your specific dataset, then copy the winning values back into the main training notebooks.

> **Compute note:** Each trial runs for `TUNE_EPOCHS` epochs. With `ITERATIONS=50` trials
> at 30 epochs each, expect ~1500 total epochs of training (several hours on RTX 3060).
> Start with fewer iterations to get a feel for the search space.

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────
# Pick ONE model + dataset to tune at a time.

# Which model to tune
MODEL       = "yolo11n.pt"              # use the same base weights as your training notebook
DATASET_DIR = "datasets/ball_tracking"  # point to whichever dataset you want to tune for

# Tuning budget
TUNE_EPOCHS = 30          # epochs per trial (short runs are fine for relative comparisons)
ITERATIONS  = 50          # number of hyperparameter candidates to evaluate
IMGSZ       = 640

# Output
RUN_PROJECT  = "runs/tune"
RUN_NAME     = "ball-tracking-tune"
WANDB_PROJECT = "tobio"   # set to "" to skip W&B

In [ ]:
import wandb
from dotenv import load_dotenv
from ultralytics import YOLO

from utils import setup_device

load_dotenv()
DEVICE = setup_device()

if WANDB_PROJECT:
    wandb.init(project=WANDB_PROJECT, name=RUN_NAME)

In [ ]:
model = YOLO(MODEL)

# model.tune() runs a genetic algorithm search over the hyperparameter space.
# It will:
#   1. Generate ITERATIONS candidate hyperparameter sets
#   2. Train each for TUNE_EPOCHS epochs
#   3. Rank by mAP50-95
#   4. Mutate the best candidates and repeat
#
# Results are saved to runs/tune/<RUN_NAME>/
#   best_hyperparameters.yaml  — copy values back into your training notebook
#   tune_results.csv           — full trial history
result = model.tune(
    data=f"{DATASET_DIR}/data.yaml",
    epochs=TUNE_EPOCHS,
    iterations=ITERATIONS,
    imgsz=IMGSZ,
    device=DEVICE,
    optimizer="AdamW",   # AdamW is recommended for tuning searches
    project=RUN_PROJECT,
    name=RUN_NAME,
    plots=True,
    save=False,          # don't save intermediate checkpoints to save disk space
    val=True,
)

In [ ]:
import yaml
import os

best_args_path = os.path.join(RUN_PROJECT, RUN_NAME, "best_hyperparameters.yaml")
if os.path.exists(best_args_path):
    with open(best_args_path) as f:
        best = yaml.safe_load(f)
    print("Best hyperparameters found:")
    for k, v in best.items():
        print(f"  {k:<20} {v}")
else:
    print(f"Results file not found at {best_args_path} — check the run output directory.")

if WANDB_PROJECT:
    wandb.finish()